<a href="https://colab.research.google.com/github/fuji-184/FML/blob/main/F_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs -o rustup-init.sh

!chmod +x rustup-init.sh

!./rustup-init.sh -y

info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
warn: Updating existing toolchain, profile choice will be ignored
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: default toolchain set to stable-x86_64-unknown-linux-gnu

  stable-x86_64-unknown-linux-gnu unchanged - rustc 1.96.0 (ac68faa20 2026-05-25)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leadin

In [ ]:
import os

os.environ["PATH"] = "/root/.cargo/bin:/usr/local/cuda/bin:" + os.environ["PATH"]
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

print("Path berhasil dikonfigurasi!")

Path berhasil dikonfigurasi!


In [ ]:
import os, torch

os.environ["LIBTORCH_USE_PYTORCH"] = "1"
os.environ["LD_LIBRARY_PATH"] = (
    os.path.dirname(torch.__file__) + "/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

In [ ]:
!rustc --version
!nvidia-smi

rustc 1.96.0 (ac68faa20 2026-05-25)
Mon Jun 29 11:50:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------

In [ ]:
!cargo new tes

    Creating binary (application) `tes` package
note: see more `Cargo.toml` keys and their definitions at https://doc.rust-lang.org/cargo/reference/manifest.html


In [ ]:
!ls tes
%cd tes

Cargo.toml  src
/content/tes/tes


In [ ]:
!cat Cargo.toml

[package]
name = "tes"
version = "0.1.0"
edition = "2024"

[dependencies]


In [ ]:

%%writefile Cargo.toml
[package]
name = "tes"
version = "0.1.0"
edition = "2024"

[dependencies]
tch = "*"
serde_json = "1"
serde = { version = "1", features = ["derive"] }
glob = "0.3"
tiktoken-rs = "*"
hf-hub = "*"
rand = "*"

Overwriting Cargo.toml


In [ ]:

%%writefile src/long_rope_position.rs
use std::f64::consts::PI;
use tch::{Kind, Tensor, Device};

pub fn compute_yarn_scaling(
    dim: i64,
    theta: f64,
    scaling_factor: f64,
    original_max_pos: i64,
    max_pos: i64,
    beta_fast: f64,
    beta_slow: f64,
    mscale: f64,
) -> (Tensor, Tensor, f64) {
    let arange_half = Tensor::arange_start_step(0, dim, 2, (Kind::Float, Device::Cpu));

    let exponents = arange_half.f_div_scalar(dim as f64).unwrap();
    let freq_extra = {
        let base: Vec<f64> = Vec::<f64>::try_from(&exponents).unwrap()
            .into_iter()
            .map(|x| 1.0 / theta.powf(x))
            .collect();
        Tensor::from_slice(&base)
    };
    let freq_inter = {
        let base: Vec<f64> = Vec::<f64>::try_from(&exponents).unwrap()
            .into_iter()
            .map(|x| 1.0 / (scaling_factor * theta.powf(x)))
            .collect();
        Tensor::from_slice(&base)
    };

    let low_raw = (dim as f64 * (original_max_pos as f64 / (beta_fast * 2.0 * PI)).ln() / (2.0 * theta.ln())).floor() as i64;
    let high_raw = (dim as f64 * (original_max_pos as f64 / (beta_slow * 2.0 * PI)).ln() / (2.0 * theta.ln())).ceil() as i64;
    let low = low_raw.max(0);
    let high = high_raw.min(dim / 2 - 1);

    let ramp_range = (high - low).max(1) as f64;
    let linear_ramp = (Tensor::arange(dim / 2, (Kind::Float, Device::Cpu)) - low as f64)
        .f_div_scalar(ramp_range)
        .unwrap()
        .clamp(0.0, 1.0);

    let inv_freq = &freq_inter * (1.0 - &linear_ramp) + &freq_extra * &linear_ramp;

    let scale_factor = if mscale == 1.0 {
        0.1 * scaling_factor.ln() + 1.0
    } else {
        mscale
    };

    (inv_freq, freq_extra, scale_factor)
}

pub struct LongRoPEEmbedding {
    pub dim: i64,
    pub theta: f64,
    pub scaling_factor: f64,
    pub original_max_pos: i64,
    pub max_pos: i64,
    pub mscale: f64,
    pub inv_freq: Tensor,
    pub cos_cache: Option<Tensor>,
    pub sin_cache: Option<Tensor>,
    pub cache_len: i64,
}

impl LongRoPEEmbedding {
    pub fn new(
        dim: i64,
        theta: f64,
        scaling_factor: f64,
        original_max_pos: i64,
        max_pos: i64,
    ) -> Self {
        let (inv_freq, _, mscale) = compute_yarn_scaling(
            dim, theta, scaling_factor, original_max_pos, max_pos, 32.0, 1.0, 1.0,
        );

        Self {
            dim,
            theta,
            scaling_factor,
            original_max_pos,
            max_pos,
            mscale,
            inv_freq,
            cos_cache: None,
            sin_cache: None,
            cache_len: 0,
        }
    }

    pub fn build_cache(&mut self, seq_len: i64, kind: Kind, device: Device) {
        if seq_len <= self.cache_len && self.cos_cache.is_some() {
            return;
        }

        let t = Tensor::arange(seq_len, (Kind::Float, device));
        let inv_freq_on_device = self.inv_freq.to(device);
        let freqs = t.unsqueeze(1).matmul(&inv_freq_on_device.unsqueeze(0));
        let emb = Tensor::cat(&[&freqs, &freqs], -1);

        self.cos_cache = Some((emb.cos() * self.mscale).to_kind(kind));
        self.sin_cache = Some((emb.sin() * self.mscale).to_kind(kind));
        self.cache_len = seq_len;
    }

    pub fn forward(&mut self, x: &Tensor, position_ids: &Tensor) -> (Tensor, Tensor) {
        let seq_len = i64::try_from(position_ids.max()).unwrap() + 1;
        self.build_cache(seq_len, x.kind(), x.device());

        let cos = self.cos_cache.as_ref().unwrap().index_select(0, &position_ids.flatten(0, -1));
        let sin = self.sin_cache.as_ref().unwrap().index_select(0, &position_ids.flatten(0, -1));

        (cos, sin)
    }
}

pub fn rotate_half(x: &Tensor) -> Tensor {
    let half = x.size().last().unwrap() / 2;
    let x1 = x.narrow(-1, 0, half);
    let x2 = x.narrow(-1, half, half);
    Tensor::cat(&[&(-x2), &x1], -1)
}

pub fn apply_rotary_emb(
    q: &Tensor,
    k: &Tensor,
    cos: &Tensor,
    sin: &Tensor,
) -> (Tensor, Tensor) {
    let cos_expanded = cos.unsqueeze(1);
    let sin_expanded = sin.unsqueeze(1);

    let q_rotated = q * &cos_expanded + rotate_half(q) * &sin_expanded;
    let k_rotated = k * &cos_expanded + rotate_half(k) * &sin_expanded;

    (q_rotated, k_rotated)
}

Writing src/long_rope_position.rs


In [ ]:

%%writefile src/model_config.rs
#[derive(Clone)]
pub struct ModelConfig {
    pub vocab_size: i64,
    pub hidden_size: i64,
    pub intermediate_size: i64,
    pub num_hidden_layers: i64,
    pub num_attention_heads: i64,
    pub num_key_value_heads: i64,
    pub head_dim: i64,

    pub kv_lora_rank: i64,
    pub q_lora_rank: i64,
    pub qk_rope_head_dim: i64,
    pub qk_nope_head_dim: i64,
    pub v_head_dim: i64,

    pub rope_theta: f64,
    pub rope_scaling_factor: f64,
    pub rope_original_max_position: i64,
    pub max_position_embeddings: i64,

    pub num_experts: i64,
    pub num_shared_experts: i64,
    pub num_experts_per_tok: i64,
    pub moe_intermediate_size: i64,
    pub moe_layer_freq: i64,
    pub first_k_dense_replace: i64,

    pub norm_eps: f64,
    pub hidden_dropout: f64,
    pub attention_dropout: f64,

    pub tie_word_embeddings: bool,
    pub initializer_range: f64,

    pub chunk_size_mb: f64,
    pub offload_to_cpu: bool,
}

impl ModelConfig {
    pub fn default() -> Self {
        Self {
            vocab_size: 102400,
            hidden_size: 2048,
            intermediate_size: 8192,
            num_hidden_layers: 28,
            num_attention_heads: 16,
            num_key_value_heads: 8,
            head_dim: 128,

            kv_lora_rank: 512,
            q_lora_rank: 768,
            qk_rope_head_dim: 64,
            qk_nope_head_dim: 64,
            v_head_dim: 128,

            rope_theta: 10000.0,
            rope_scaling_factor: 40.0,
            rope_original_max_position: 4096,
            max_position_embeddings: 131072,

            num_experts: 64,
            num_shared_experts: 2,
            num_experts_per_tok: 6,
            moe_intermediate_size: 1024,
            moe_layer_freq: 1,
            first_k_dense_replace: 1,

            norm_eps: 1e-6,
            hidden_dropout: 0.0,
            attention_dropout: 0.0,

            tie_word_embeddings: false,
            initializer_range: 0.02,

            chunk_size_mb: 500.0,
            offload_to_cpu: true,
        }
    }

    pub fn total_params_approx(&self) -> String {
        let embedding_params = self.vocab_size * self.hidden_size;

        let attention_params_per_layer = self.q_lora_rank * self.hidden_size
            + self.kv_lora_rank * self.hidden_size
            + self.num_attention_heads * self.qk_rope_head_dim
            + (self.num_attention_heads * self.qk_nope_head_dim
                + self.num_key_value_heads * self.qk_nope_head_dim)
                * self.kv_lora_rank
            + self.num_attention_heads * self.v_head_dim * self.hidden_size;

        let moe_params_per_layer = self.num_experts * 2 * self.hidden_size * self.moe_intermediate_size
            + self.num_shared_experts * 2 * self.hidden_size * self.intermediate_size;

        let total = embedding_params
            + self.num_hidden_layers * (attention_params_per_layer + moe_params_per_layer);

        format!("~{:.1}B", total as f64 / 1e9)
    }
}

pub fn preset_4b() -> ModelConfig {
    ModelConfig {
        vocab_size: 102400,
        hidden_size: 2048,
        intermediate_size: 8192,
        num_hidden_layers: 28,
        num_attention_heads: 16,
        num_key_value_heads: 8,
        head_dim: 128,
        kv_lora_rank: 512,
        q_lora_rank: 768,
        qk_rope_head_dim: 64,
        qk_nope_head_dim: 64,
        v_head_dim: 128,
        rope_theta: 10000.0,
        rope_scaling_factor: 40.0,
        rope_original_max_position: 4096,
        max_position_embeddings: 131072,
        num_experts: 64,
        num_shared_experts: 2,
        num_experts_per_tok: 6,
        moe_intermediate_size: 1024,
        moe_layer_freq: 1,
        first_k_dense_replace: 1,
        chunk_size_mb: 500.0,
        ..ModelConfig::default()
    }
}

Writing src/model_config.rs


In [ ]:


%%writefile src/hybrid_mla.rs
use tch::{nn, Tensor, Kind, Device};
use tch::nn::{Module, LinearConfig};
use crate::long_rope_position::LongRoPEEmbedding;
use crate::model_config::ModelConfig;
use crate::long_rope_position::apply_rotary_emb;
use crate::model::RmsNorm;

pub struct MLAAttention {
    pub layer_idx: i64,
    pub hidden_size: i64,
    pub num_heads: i64,
    pub num_kv_heads: i64,
    pub head_dim: i64,
    pub kv_lora_rank: i64,
    pub q_lora_rank: i64,
    pub qk_rope_head_dim: i64,
    pub qk_nope_head_dim: i64,
    pub v_head_dim: i64,
    pub attention_dropout: f64,
    pub kv_groups: i64,

    pub q_a_proj: nn::Linear,
    pub q_a_layernorm: RmsNorm,
    pub q_b_proj: nn::Linear,

    pub kv_a_proj_with_mqa: nn::Linear,
    pub kv_a_layernorm: RmsNorm,
    pub kv_b_proj: nn::Linear,

    pub o_proj: nn::Linear,

    pub rope: LongRoPEEmbedding,
}

impl MLAAttention {
    pub fn new(vs: &nn::Path, config: &ModelConfig, layer_idx: i64) -> Self {
        let no_bias = LinearConfig { bias: false, ..Default::default() };

        let q_a_proj = nn::linear(
            vs / "q_a_proj",
            config.hidden_size,
            config.q_lora_rank,
            no_bias,
        );
        let q_a_layernorm = RmsNorm::new(
            vs / "q_a_layernorm",
            config.q_lora_rank,
            config.norm_eps,
        );
        let q_b_proj = nn::linear(
            vs / "q_b_proj",
            config.q_lora_rank,
            config.num_attention_heads * (config.qk_nope_head_dim + config.qk_rope_head_dim),
            no_bias,
        );

        let kv_a_proj_with_mqa = nn::linear(
            vs / "kv_a_proj_with_mqa",
            config.hidden_size,
            config.kv_lora_rank + config.qk_rope_head_dim,
            no_bias,
        );
        let kv_a_layernorm = RmsNorm::new(
            vs / "kv_a_layernorm",
            config.kv_lora_rank,
            config.norm_eps,
        );
        let kv_b_proj = nn::linear(
            vs / "kv_b_proj",
            config.kv_lora_rank,
            config.num_key_value_heads * (config.qk_nope_head_dim + config.v_head_dim),
            no_bias,
        );

        let o_proj = nn::linear(
            vs / "o_proj",
            config.num_attention_heads * config.v_head_dim,
            config.hidden_size,
            no_bias,
        );

        let rope = LongRoPEEmbedding::new(
            config.qk_rope_head_dim,
            config.rope_theta,
            config.rope_scaling_factor,
            config.rope_original_max_position,
            config.max_position_embeddings,
        );

        Self {
            layer_idx,
            hidden_size: config.hidden_size,
            num_heads: config.num_attention_heads,
            num_kv_heads: config.num_key_value_heads,
            head_dim: config.head_dim,
            kv_lora_rank: config.kv_lora_rank,
            q_lora_rank: config.q_lora_rank,
            qk_rope_head_dim: config.qk_rope_head_dim,
            qk_nope_head_dim: config.qk_nope_head_dim,
            v_head_dim: config.v_head_dim,
            attention_dropout: config.attention_dropout,
            kv_groups: config.num_attention_heads / config.num_key_value_heads,
            q_a_proj,
            q_a_layernorm,
            q_b_proj,
            kv_a_proj_with_mqa,
            kv_a_layernorm,
            kv_b_proj,
            o_proj,
            rope,
        }
    }

    pub fn sparse_causal_mask(&self, attn_weights: &Tensor, local_window: i64) -> Tensor {
        let shape = attn_weights.size();
        let (tgt_len, src_len) = (shape[2], shape[3]);

        let mut mask = Tensor::full(
            &[tgt_len, src_len],
            f64::NEG_INFINITY,
            (attn_weights.kind(), attn_weights.device()),
        );
        mask = mask.triu(1);

        if tgt_len > local_window {
            for i in 0..tgt_len {
                let start = (i - local_window).max(0);
                if start > 0 {
                    let _ = mask.narrow(0, i, 1).narrow(1, 0, start).fill_(f64::NEG_INFINITY);
                }
            }
        }

        attn_weights + mask.unsqueeze(0).unsqueeze(0)
    }

    pub fn forward(
        &mut self,
        hidden_states: &Tensor,
        position_ids: &Tensor,
        attention_mask: Option<&Tensor>,
        past_key_value: Option<(Tensor, Tensor)>,
        use_cache: bool,
        use_sparse: bool,
    ) -> (Tensor, Option<(Tensor, Tensor)>) {
        let shape = hidden_states.size();
        let (bsz, q_len) = (shape[0], shape[1]);

        let q = self.q_b_proj.forward(&self.q_a_layernorm.forward(&self.q_a_proj.forward(hidden_states)));
        let q = q.view([bsz, q_len, self.num_heads, self.qk_nope_head_dim + self.qk_rope_head_dim]);
        let q_nope = q.narrow(-1, 0, self.qk_nope_head_dim);
        let q_rope = q.narrow(-1, self.qk_nope_head_dim, self.qk_rope_head_dim);

        let kv_a = self.kv_a_proj_with_mqa.forward(hidden_states);
        let kv_a_latent = kv_a.narrow(-1, 0, self.kv_lora_rank);
        let k_rope_raw = kv_a.narrow(-1, self.kv_lora_rank, self.qk_rope_head_dim);

        let kv_a_latent = self.kv_a_layernorm.forward(&kv_a_latent);
        let kv = self.kv_b_proj.forward(&kv_a_latent);
        let kv = kv.view([bsz, q_len, self.num_kv_heads, self.qk_nope_head_dim + self.v_head_dim]);
        let k_nope = kv.narrow(-1, 0, self.qk_nope_head_dim);
        let v = kv.narrow(-1, self.qk_nope_head_dim, self.v_head_dim);

        let k_rope_raw = k_rope_raw
            .view([bsz, q_len, 1, self.qk_rope_head_dim])
            .expand(&[bsz, q_len, self.num_kv_heads, self.qk_rope_head_dim], false);

        let (cos, sin) = self.rope.forward(hidden_states, position_ids);

        let (q_rope_rot, k_rope_rot) = apply_rotary_emb(
            &q_rope.transpose(1, 2),
            &k_rope_raw.transpose(1, 2),
            &cos,
            &sin,
        );
        let q_rope_rot = q_rope_rot.transpose(1, 2);
        let k_rope_rot = k_rope_rot.transpose(1, 2);

        let q_full = Tensor::cat(&[&q_nope, &q_rope_rot], -1).transpose(1, 2);
        let k_nope_expanded = k_nope.repeat_interleave_self_int(self.kv_groups, 2, None);
        let k_rope_expanded = k_rope_rot.repeat_interleave_self_int(self.kv_groups, 2, None);
        let k_full = Tensor::cat(&[&k_nope_expanded, &k_rope_expanded], -1).transpose(1, 2);
        let v = v.transpose(1, 2);

        let (k_full, v) = if let Some((past_k, past_v)) = past_key_value {
            (
                Tensor::cat(&[&past_k, &k_full], 2),
                Tensor::cat(&[&past_v, &v], 2),
            )
        } else {
            (k_full, v)
        };

        let new_past = if use_cache {
            Some((k_full.shallow_clone(), v.shallow_clone()))
        } else {
            None
        };

        let k_head_dim = self.qk_nope_head_dim + self.qk_rope_head_dim;
        let scale = (k_head_dim as f64).powf(-0.5);
        let mut attn_weights = q_full.matmul(&k_full.transpose(-2, -1)) * scale;

        attn_weights = if use_sparse && q_len > 1 {
            self.sparse_causal_mask(&attn_weights, 2048)
        } else if let Some(mask) = attention_mask {
            attn_weights + mask
        } else {
            attn_weights
        };

        let mut attn_weights = attn_weights
            .to_kind(Kind::Float)
            .softmax(-1, Kind::Float)
            .to_kind(q_full.kind());

        if self.attention_dropout > 0.0 {
            attn_weights = attn_weights.dropout(self.attention_dropout, true);
        }

        let v_expanded = v.repeat_interleave_self_int(self.kv_groups, 1, None);
        let attn_output = attn_weights.matmul(&v_expanded);

        let attn_output = attn_output
            .transpose(1, 2)
            .contiguous()
            .view([bsz, q_len, self.num_heads * self.v_head_dim]);
        let attn_output = self.o_proj.forward(&attn_output);

        (attn_output, new_past)
    }
}

Writing src/hybrid_mla.rs


In [ ]:

%%writefile src/expert.rs
use tch::{nn, Tensor, Kind};
use tch::nn::{Module, LinearConfig};
use crate::model_config::ModelConfig;

pub struct Expert {
    pub gate_proj: nn::Linear,
    pub up_proj: nn::Linear,
    pub down_proj: nn::Linear,
}

impl Expert {
    pub fn new(vs: &nn::Path, hidden_size: i64, intermediate_size: i64) -> Self {
        let no_bias = LinearConfig { bias: false, ..Default::default() };
        Self {
            gate_proj: nn::linear(vs / "gate_proj", hidden_size, intermediate_size, no_bias),
            up_proj: nn::linear(vs / "up_proj", hidden_size, intermediate_size, no_bias),
            down_proj: nn::linear(vs / "down_proj", intermediate_size, hidden_size, no_bias),
        }
    }

    pub fn forward(&self, x: &Tensor) -> Tensor {
        let gate = self.gate_proj.forward(x).silu();
        let up = self.up_proj.forward(x);
        self.down_proj.forward(&(gate * up))
    }
}

pub struct RouterGating {
    pub gate: nn::Linear,
    pub num_experts: i64,
    pub top_k: i64,
}

impl RouterGating {
    pub fn new(vs: &nn::Path, hidden_size: i64, num_experts: i64, top_k: i64) -> Self {
        let no_bias = LinearConfig { bias: false, ..Default::default() };
        Self {
            gate: nn::linear(vs / "gate", hidden_size, num_experts, no_bias),
            num_experts,
            top_k,
        }
    }

    pub fn forward(&self, hidden_states: &Tensor) -> (Tensor, Tensor, Tensor) {
        let scores = self.gate.forward(hidden_states).softmax(-1, Kind::Float);
        let (top_k_weights, top_k_indices) = scores.topk(self.top_k, -1, true, true);
        let top_k_weights = top_k_weights / top_k_weights.sum_dim_intlist(&[-1i64][..], true, Kind::Float);
        (top_k_weights, top_k_indices, scores)
    }
}

pub struct HybridSparseMoE {
    pub num_experts: i64,
    pub num_shared: i64,
    pub top_k: i64,
    pub hidden_size: i64,
    pub router: RouterGating,
    pub routed_experts: Vec<Expert>,
    pub shared_experts: Vec<Expert>,
    pub shared_weight: Tensor,
}

impl HybridSparseMoE {
    pub fn new(vs: &nn::Path, config: &ModelConfig) -> Self {
        let router = RouterGating::new(
            &(vs / "router"),
            config.hidden_size,
            config.num_experts,
            config.num_experts_per_tok,
        );

        let routed_experts = (0..config.num_experts)
            .map(|i| Expert::new(&(vs / "routed_experts" / i), config.hidden_size, config.moe_intermediate_size))
            .collect();

        let shared_experts = (0..config.num_shared_experts)
            .map(|i| Expert::new(&(vs / "shared_experts" / i), config.hidden_size, config.intermediate_size))
            .collect();

        let shared_weight = vs.var(
            "shared_weight",
            &[config.num_shared_experts],
            nn::Init::Const(1.0 / config.num_shared_experts as f64),
        );

        Self {
            num_experts: config.num_experts,
            num_shared: config.num_shared_experts,
            top_k: config.num_experts_per_tok,
            hidden_size: config.hidden_size,
            router,
            routed_experts,
            shared_experts,
            shared_weight,
        }
    }

    pub fn compute_aux_loss(&self, scores: &Tensor, top_k_indices: &Tensor) -> Tensor {
        let bsz_seq = scores.size()[0];
        let mut expert_counts = Tensor::zeros(
            &[self.num_experts],
            (scores.kind(), scores.device()),
        );
        let ones = Tensor::ones(
            &[bsz_seq, self.top_k],
            (scores.kind(), scores.device()),
        );
        expert_counts.scatter_add_(0, &top_k_indices.view([-1]), &ones.view([-1]));
        let fraction = &expert_counts / (bsz_seq * self.top_k) as f64;
        let router_prob = scores.mean_dim(&[0i64][..], false, scores.kind());
        (&fraction * &router_prob).sum(Kind::Float) * self.num_experts as f64
    }

    pub fn forward(&self, hidden_states: &Tensor) -> (Tensor, Tensor) {
        let shape = hidden_states.size();
        let (bsz, seq_len, d) = (shape[0], shape[1], shape[2]);
        let flat = hidden_states.view([-1, d]);
        let num_tokens = flat.size()[0];

        let (top_k_weights, top_k_indices, scores) = self.router.forward(&flat);
        let aux_loss = self.compute_aux_loss(&scores, &top_k_indices);

        let mut routed_out = Tensor::zeros_like(&flat);

        let top_k_indices_flat = top_k_indices.view([-1]);
        let top_k_indices_vec: Vec<i64> = top_k_indices_flat.iter::<i64>().unwrap().collect();
        let mut expert_to_tokens: Vec<Vec<(i64, i64)>> = vec![vec![]; self.num_experts as usize];
        for tok_idx in 0..num_tokens {
            for k in 0..self.top_k {
                let eid = top_k_indices_vec[(tok_idx * self.top_k + k) as usize] as usize;
                expert_to_tokens[eid].push((tok_idx, k));
            }
        }

        for (eid, assignments) in expert_to_tokens.iter().enumerate() {
            if assignments.is_empty() {
                continue;
            }
            let tok_ids_vec: Vec<i64> = assignments.iter().map(|(t, _)| *t).collect();
            let k_ids_vec: Vec<i64> = assignments.iter().map(|(_, k)| *k).collect();

            let tok_ids = Tensor::from_slice(&tok_ids_vec).to(flat.device());
            let k_ids = Tensor::from_slice(&k_ids_vec).to(flat.device());

            let expert_input = flat.index_select(0, &tok_ids);
            let expert_output = self.routed_experts[eid].forward(&expert_input);

            let weights = top_k_weights
                .index_select(0, &tok_ids)
                .gather(1, &k_ids.unsqueeze(-1), false)
                .unsqueeze(-1);

            let weighted_output = expert_output * weights;
            let scatter_index = tok_ids.unsqueeze(-1).expand_as(&weighted_output);
            routed_out.scatter_add_(0, &scatter_index, &weighted_output);
        }

        let shared_weights = self.shared_weight.softmax(-1, Kind::Float);
        let mut shared_out = Tensor::zeros_like(&flat);
        for (i, expert) in self.shared_experts.iter().enumerate() {
            shared_out = shared_out + shared_weights.get(i as i64) * expert.forward(&flat);
        }

        let output = (routed_out + shared_out).view([bsz, seq_len, d]);
        (output, aux_loss)
    }
}

Writing src/expert.rs


In [ ]:

%%writefile src/model.rs
use tch::{nn, Tensor, Kind, Device};use tch::nn::{Module, LinearConfig, EmbeddingConfig};use crate::hybrid_mla::MLAAttention;use crate::expert::{HybridSparseMoE, Expert};use crate::model_config::ModelConfig;
pub struct RmsNorm {    pub weight: Tensor,    pub eps: f64,}
impl RmsNorm {    pub fn new(vs: nn::Path, size: i64, eps: f64) -> Self {        let weight = vs.ones("weight", &[size]);        Self { weight, eps }    }}
impl Module for RmsNorm {    fn forward(&self, xs: &Tensor) -> Tensor {        let kind = xs.kind();        let xs_f32 = xs.to_kind(Kind::Float);        let rms = (&xs_f32 * &xs_f32).mean_dim(&[-1i64][..], true, Kind::Float);        let xs_normed = xs_f32 / (rms + self.eps).sqrt();        (xs_normed.to_kind(kind) * &self.weight)    }}
pub enum MlpLayer {    MoE(HybridSparseMoE),    Dense(Expert),}
pub struct DecoderLayer {    pub layer_idx: i64,    pub self_attn: MLAAttention,    pub mlp: MlpLayer,    pub input_layernorm: RmsNorm,    pub post_attention_layernorm: RmsNorm,}
impl DecoderLayer {    pub fn new(vs: &nn::Path, config: &ModelConfig, layer_idx: i64) -> Self {        let self_attn = MLAAttention::new(&(vs / "self_attn"), config, layer_idx);
        let use_moe = layer_idx >= config.first_k_dense_replace            && (layer_idx - config.first_k_dense_replace) % config.moe_layer_freq == 0;
        let mlp = if use_moe {            MlpLayer::MoE(HybridSparseMoE::new(&(vs / "mlp"), config))        } else {            MlpLayer::Dense(Expert::new(&(vs / "mlp"), config.hidden_size, config.intermediate_size))        };
        let input_layernorm = RmsNorm::new(            vs / "input_layernorm",            config.hidden_size,            config.norm_eps,        );        let post_attention_layernorm = RmsNorm::new(            vs / "post_attention_layernorm",            config.hidden_size,            config.norm_eps,        );
        Self {            layer_idx,            self_attn,            mlp,            input_layernorm,            post_attention_layernorm,        }    }
    pub fn forward(        &mut self,        hidden_states: &Tensor,        position_ids: &Tensor,        attention_mask: Option<&Tensor>,        past_key_value: Option<(Tensor, Tensor)>,        use_cache: bool,    ) -> (Tensor, Option<(Tensor, Tensor)>, Option<Tensor>) {        let residual = hidden_states.shallow_clone();        let hidden_states = self.input_layernorm.forward(hidden_states);
        let (attn_out, new_past) = self.self_attn.forward(            &hidden_states,            position_ids,            attention_mask,            past_key_value,            use_cache,            true,        );        let hidden_states = residual + attn_out;
        let residual = hidden_states.shallow_clone();        let hidden_states = self.post_attention_layernorm.forward(&hidden_states);
        let (hidden_states, aux_loss) = match &self.mlp {            MlpLayer::MoE(moe) => {                let (out, loss) = moe.forward(&hidden_states);                (out, Some(loss))            }            MlpLayer::Dense(expert) => {                (expert.forward(&hidden_states), None)            }        };
        let hidden_states = residual + hidden_states;        (hidden_states, new_past, aux_loss)    }}

pub struct CustomLLM {    pub config: ModelConfig,    pub embed_tokens: nn::Embedding,    pub layers: Vec<DecoderLayer>,    pub norm: RmsNorm,    pub lm_head: Option<nn::Linear>,}
impl CustomLLM {    pub fn new(vs: &nn::Path, config: ModelConfig) -> Self {        let embed_tokens = nn::embedding(            vs / "embed_tokens",            config.vocab_size,            config.hidden_size,            EmbeddingConfig::default(),        );
        let layers = (0..config.num_hidden_layers)            .map(|i| DecoderLayer::new(&(vs / "layers" / i), &config, i))            .collect();
        let norm = RmsNorm::new(            vs / "norm",            config.hidden_size,            config.norm_eps,        );
        let lm_head = if config.tie_word_embeddings {            None        } else {            let no_bias = LinearConfig { bias: false, ..Default::default() };            Some(nn::linear(vs / "lm_head", config.hidden_size, config.vocab_size, no_bias))        };
        let model = Self {            config,            embed_tokens,            layers,            norm,            lm_head,        };
        model    }
    pub fn forward(        &mut self,        input_ids: &Tensor,        position_ids: Option<&Tensor>,        attention_mask: Option<&Tensor>,        past_key_values: Option<&Vec<Option<(Tensor, Tensor)>>>,        use_cache: bool,        return_aux_loss: bool,    ) -> (Tensor, Option<Vec<Option<(Tensor, Tensor)>>>, Option<Tensor>) {        let shape = input_ids.size();        let (_bsz, seq_len) = (shape[0], shape[1]);
        let position_ids = match position_ids {            Some(p) => p.shallow_clone(),            None => {                let offset = past_key_values                    .and_then(|pkvs| pkvs.first())                    .and_then(|pkv| pkv.as_ref())                    .map(|(k, _)| k.size()[2])                    .unwrap_or(0);                Tensor::arange_start(offset, offset + seq_len, (Kind::Int64, input_ids.device()))                    .unsqueeze(0)            }        };
        let mut hidden_states = self.embed_tokens.forward(input_ids);
        let mut new_past: Option<Vec<Option<(Tensor, Tensor)>>> = if use_cache {            Some(Vec::new())        } else {            None        };
        let mut total_aux_loss = Tensor::zeros(&[], (Kind::Float, input_ids.device()));
        for (i, layer) in self.layers.iter_mut().enumerate() {            let past = past_key_values                .and_then(|pkvs| pkvs.get(i))                .and_then(|pkv| pkv.as_ref())                .map(|(k, v)| (k.shallow_clone(), v.shallow_clone()));
            let (next_hidden, layer_past, aux_loss) = layer.forward(                &hidden_states,                &position_ids,                attention_mask,                past,                use_cache,            );
            hidden_states = next_hidden;
            if let Some(ref mut cache) = new_past {                cache.push(layer_past);            }
            if return_aux_loss {                if let Some(loss) = aux_loss {                    total_aux_loss = total_aux_loss + loss;                }            }        }
        let hidden_states = self.norm.forward(&hidden_states);
        let logits = match &self.lm_head {            Some(head) => head.forward(&hidden_states),            None => hidden_states.matmul(&self.embed_tokens.ws.transpose(0, 1)),        };
        let aux_loss_out = if return_aux_loss {            Some(total_aux_loss)        } else {            None        };
        (logits, new_past, aux_loss_out)    }}

Writing src/model.rs


In [ ]:

%%writefile src/util.rs
use std::collections::HashMap;
use tch::{nn, Tensor, Kind};
use tch::nn::OptimizerConfig;
use crate::model::CustomLLM;
use crate::model_config::ModelConfig;

pub fn compute_loss(
    model: &mut CustomLLM,
    input_ids: &Tensor,
    labels: &Tensor,
    aux_loss_coef: f64,
) -> (Tensor, HashMap<String, f64>) {
    let (logits, _, aux_loss) = model.forward(input_ids, None, None, None, false, true);

    let shift_logits = logits.narrow(1, 0, logits.size()[1] - 1).contiguous();
    let shift_labels = labels.narrow(1, 1, labels.size()[1] - 1).contiguous();

    let vocab_size = shift_logits.size()[2];
    let lm_loss = shift_logits
        .view([-1, vocab_size])
        .cross_entropy_loss(
            &shift_labels.view([-1]),
            None::<&Tensor>,
            tch::Reduction::Mean,
            -100,
            0.0,
        );

    let aux = aux_loss.unwrap_or_else(|| Tensor::zeros(&[], (Kind::Float, input_ids.device())));
    let total_loss = &lm_loss + aux_loss_coef * &aux;

    let mut metrics = HashMap::new();
    metrics.insert("lm_loss".to_string(), f64::try_from(&lm_loss).unwrap());
    metrics.insert("aux_loss".to_string(), f64::try_from(&aux).unwrap());
    metrics.insert("total_loss".to_string(), f64::try_from(&total_loss).unwrap());

    (total_loss, metrics)
}


pub fn build_optimizer(
    vs: &nn::VarStore,
    lr: f64,
    weight_decay: f64,
    beta1: f64,
    beta2: f64,
) -> nn::Optimizer {
    nn::AdamW::default()
        .beta1(beta1)
        .beta2(beta2)
        .wd(weight_decay)
        .build(vs, lr)
        .unwrap()
}


pub struct LRScheduler {
    pub warmup_steps: usize,
    pub max_steps: usize,
    pub min_lr_ratio: f64,
    pub base_lr: f64,
}

impl LRScheduler {
    pub fn new(base_lr: f64, max_steps: usize, warmup_steps: usize, min_lr_ratio: f64) -> Self {
        Self {
            warmup_steps,
            max_steps,
            min_lr_ratio,
            base_lr,
        }
    }

    pub fn get_lr(&self, step: usize) -> f64 {
        let scale = if step < self.warmup_steps {
            step as f64 / self.warmup_steps.max(1) as f64
        } else {
            let progress = (step - self.warmup_steps) as f64
                / (self.max_steps - self.warmup_steps).max(1) as f64;
            let cosine = 0.5 * (1.0 + (std::f64::consts::PI * progress).cos());
            self.min_lr_ratio + (1.0 - self.min_lr_ratio) * cosine
        };
        self.base_lr * scale
    }

    pub fn step(&self, optimizer: &mut nn::Optimizer, current_step: usize) {
        optimizer.set_lr(self.get_lr(current_step));
    }
}


pub fn count_parameters(vs: &nn::VarStore) -> HashMap<String, f64> {
    let mut total: i64 = 0;
    let mut trainable: i64 = 0;

    for (_, tensor) in vs.variables() {
        let numel: i64 = tensor.size().iter().product();
        total += numel;
        if tensor.requires_grad() {
            trainable += numel;
        }
    }

    let mut result = HashMap::new();
    result.insert("total".to_string(), total as f64);
    result.insert("trainable".to_string(), trainable as f64);
    result.insert("total_B".to_string(), total as f64 / 1e9);
    result.insert("trainable_B".to_string(), trainable as f64 / 1e9);
    result
}


pub fn estimate_active_params(config: &ModelConfig) -> HashMap<String, f64> {
    let embedding_params = config.vocab_size * config.hidden_size;

    let attention_params_per_layer = config.q_lora_rank * config.hidden_size
        + config.kv_lora_rank * config.hidden_size
        + config.num_attention_heads
            * (config.qk_nope_head_dim + config.qk_rope_head_dim)
            * config.kv_lora_rank
        + config.num_attention_heads * config.v_head_dim * config.hidden_size;

    let shared_mlp_params = config.num_shared_experts * 3 * config.hidden_size * config.intermediate_size;
    let routed_mlp_active_params = config.num_experts_per_tok * 3 * config.hidden_size * config.moe_intermediate_size;
    let moe_active_per_layer = shared_mlp_params + routed_mlp_active_params;
    let dense_active_per_layer = 3 * config.hidden_size * config.intermediate_size;

    let mut active = embedding_params;
    for i in 0..config.num_hidden_layers {
        active += attention_params_per_layer;
        let use_moe = i >= config.first_k_dense_replace
            && (i - config.first_k_dense_replace) % config.moe_layer_freq == 0;
        active += if use_moe { moe_active_per_layer } else { dense_active_per_layer };
    }

    let mut result = HashMap::new();
    result.insert("active_params".to_string(), active as f64);
    result.insert("active_params_B".to_string(), active as f64 / 1e9);
    result
}

Writing src/util.rs


In [ ]:
%%writefile src/saver.rs
use std::collections::HashMap;
use std::fs::{self, File};
use std::io::{BufWriter, Write};
use std::path::{Path, PathBuf};
use serde_json::{json, Value};
use tch::{Kind, Tensor};

const MB: f64 = (1024 * 1024) as f64;

pub fn dtype_to_str(kind: Kind) -> &'static str {
    match kind {
        Kind::Float => "F32",
        Kind::Half => "F16",
        Kind::BFloat16 => "BF16",
        Kind::Int64 => "I64",
        Kind::Int => "I32",
        Kind::Int8 => "I8",
        _ => "F32",
    }
}

pub fn save_safetensors(
    tensors: &HashMap<String, Tensor>,
    path: &Path,
    metadata: Option<&HashMap<String, String>>,
) -> std::io::Result<()> {
    if let Some(parent) = path.parent() {
        fs::create_dir_all(parent)?;
    }

    let mut header_map: serde_json::Map<String, Value> = serde_json::Map::new();

    if let Some(meta) = metadata {
        let meta_obj: serde_json::Map<String, Value> = meta
            .iter()
            .map(|(k, v)| (k.clone(), Value::String(v.clone())))
            .collect();
        header_map.insert("__metadata__".to_string(), Value::Object(meta_obj));
    }

    let mut data_parts: Vec<Vec<u8>> = Vec::new();
    let mut offset: usize = 0;

    for (name, tensor) in tensors {
        let contiguous = tensor.contiguous().to_device(tch::Device::Cpu);
        let num_bytes = contiguous.numel() * contiguous.kind().elt_size_in_bytes();
        let raw = unsafe {
            let ptr = contiguous.data_ptr() as *const u8;
            std::slice::from_raw_parts(ptr, num_bytes).to_vec()
        };

        let shape: Vec<Value> = contiguous.size().iter().map(|&d| json!(d)).collect();
        header_map.insert(
            name.clone(),
            json!({
                "dtype": dtype_to_str(contiguous.kind()),
                "shape": shape,
                "data_offsets": [offset, offset + raw.len()],
            }),
        );

        offset += raw.len();
        data_parts.push(raw);
    }

    let header_bytes = serde_json::to_vec(&Value::Object(header_map)).unwrap();
    let header_len = header_bytes.len() as u64;

    let file = File::create(path)?;
    let mut writer = BufWriter::new(file);
    writer.write_all(&header_len.to_le_bytes())?;
    writer.write_all(&header_bytes)?;
    for part in &data_parts {
        writer.write_all(part)?;
    }

    Ok(())
}


pub fn save_model_sharded(
    vs: &tch::nn::VarStore,
    output_dir: &Path,
    max_shard_mb: f64,
    target_kind: Option<Kind>,
    verbose: bool,
) -> std::io::Result<()> {
    fs::create_dir_all(output_dir)?;

    let max_bytes = (max_shard_mb * MB) as usize;
    let mut shard_idx: usize = 0;
    let mut current_shard: HashMap<String, Tensor> = HashMap::new();
    let mut current_bytes: usize = 0;
    let mut shard_map: HashMap<String, String> = HashMap::new();
    let mut total_size: usize = 0;

    let flush_shard = |shard_idx: usize,
                       current_shard: &HashMap<String, Tensor>,
                       current_bytes: usize,
                       output_dir: &Path,
                       verbose: bool| -> std::io::Result<()> {
        if current_shard.is_empty() {
            return Ok(());
        }
        let fname = format!("model-{:05}.safetensors", shard_idx);
        let fpath = output_dir.join(&fname);
        save_safetensors(current_shard, &fpath, None)?;
        if verbose {
            println!(
                "  Saved shard {} ({:.1} MB, {} tensors)",
                fname,
                current_bytes as f64 / MB,
                current_shard.len(),
            );
        }
        Ok(())
    };

    for (name, tensor) in vs.variables() {
        let t = tensor.detach();
        let t = match target_kind {
            Some(kind) => t.to_kind(kind),
            None => t,
        };
        let t = t.to_device(tch::Device::Cpu);
        let size = t.numel() * t.kind().elt_size_in_bytes();

        if current_bytes + size > max_bytes && !current_shard.is_empty() {
            flush_shard(shard_idx, &current_shard, current_bytes, output_dir, verbose)?;
            shard_idx += 1;
            current_shard.clear();
            current_bytes = 0;
        }

        shard_map.insert(name.clone(), format!("model-{:05}.safetensors", shard_idx));
        current_shard.insert(name, t);
        current_bytes += size;
        total_size += size;
    }

    flush_shard(shard_idx, &current_shard, current_bytes, output_dir, verbose)?;
    shard_idx += 1;

    let index = json!({
        "metadata": { "total_size": total_size },
        "weight_map": shard_map,
    });

    let index_path = output_dir.join("model.safetensors.index.json");
    let index_file = File::create(&index_path)?;
    serde_json::to_writer_pretty(BufWriter::new(index_file), &index)?;

    if verbose {
        println!(
            "\nSaved {} shards, total {:.1} MB → {}",
            shard_idx,
            total_size as f64 / MB,
            output_dir.display(),
        );
    }

    Ok(())
}

Writing src/saver.rs


In [ ]:
%%writefile src/streaming.rs
use std::collections::{HashMap, HashSet};
use std::fs::File;
use std::io::{BufReader, Read, Seek, SeekFrom};
use std::path::{Path, PathBuf};
use std::sync::{Arc, Mutex};
use serde_json::Value;
use tch::{Kind, Tensor, Device};
use crate::model_config::ModelConfig;
use crate::model::CustomLLM;

const MB: f64 = (1024 * 1024) as f64;

pub fn tensor_size_bytes(t: &Tensor) -> usize {
    t.numel() * t.kind().elt_size_in_bytes()
}


pub struct ChunkBudget {
    pub max_bytes: usize,
    pub used: Arc<Mutex<usize>>,
}

impl ChunkBudget {
    pub fn new(max_bytes: usize) -> Self {
        Self {
            max_bytes,
            used: Arc::new(Mutex::new(0)),
        }
    }

    pub fn can_fit(&self, size_bytes: usize) -> bool {
        let used = self.used.lock().unwrap();
        *used + size_bytes <= self.max_bytes
    }

    pub fn consume(&self, size_bytes: usize) {
        let mut used = self.used.lock().unwrap();
        *used += size_bytes;
    }

    pub fn release(&self, size_bytes: usize) {
        let mut used = self.used.lock().unwrap();
        *used = used.saturating_sub(size_bytes);
    }

    pub fn reset(&self) {
        let mut used = self.used.lock().unwrap();
        *used = 0;
    }

    pub fn used_mb(&self) -> f64 {
        *self.used.lock().unwrap() as f64 / MB
    }

    pub fn max_mb(&self) -> f64 {
        self.max_bytes as f64 / MB
    }
}


pub fn str_to_kind(dtype_str: &str) -> Kind {
    match dtype_str {
        "F32" => Kind::Float,
        "F16" => Kind::Half,
        "BF16" => Kind::BFloat16,
        "I64" => Kind::Int64,
        "I32" => Kind::Int,
        "I8" => Kind::Int8,
        _ => Kind::Float,
    }
}


pub struct SafetensorsStreamReader {
    pub path: PathBuf,
    pub header: HashMap<String, Value>,
    pub data_offset: u64,
}

impl SafetensorsStreamReader {
    pub fn new(path: &Path) -> std::io::Result<Self> {
        let mut file = File::open(path)?;
        let mut size_buf = [0u8; 8];
        file.read_exact(&mut size_buf)?;
        let header_size = u64::from_le_bytes(size_buf);

        let mut header_raw = vec![0u8; header_size as usize];
        file.read_exact(&mut header_raw)?;
        let header: HashMap<String, Value> = serde_json::from_slice(&header_raw).unwrap();

        let data_offset = 8 + header_size;

        Ok(Self {
            path: path.to_path_buf(),
            header,
            data_offset,
        })
    }

    pub fn iter_tensors(
        &self,
        dtype_map: &HashMap<String, Kind>,
    ) -> std::io::Result<Vec<(String, Tensor)>> {
        let mut file = BufReader::new(File::open(&self.path)?);
        let mut results = Vec::new();

        for (name, meta) in &self.header {
            if name == "__metadata__" {
                continue;
            }
            let dtype_str = meta["dtype"].as_str().unwrap_or("F32");
            let shape: Vec<i64> = meta["shape"]
                .as_array()
                .unwrap()
                .iter()
                .map(|v| v.as_i64().unwrap())
                .collect();
            let start = meta["data_offsets"][0].as_u64().unwrap();
            let end = meta["data_offsets"][1].as_u64().unwrap();

            let base_kind = str_to_kind(dtype_str);
            file.seek(SeekFrom::Start(self.data_offset + start))?;
            let byte_count = (end - start) as usize;
            let mut raw = vec![0u8; byte_count];
            file.read_exact(&mut raw)?;

            let tensor = unsafe {
                let t = Tensor::from_blob(
                    raw.as_ptr(),
                    &shape,
                    &[],
                    base_kind,
                    Device::Cpu,
                );
                t.shallow_clone()
            };
            let tensor = tensor.contiguous();

            let target_kind = dtype_map.get(name).copied().unwrap_or(base_kind);
            let tensor = if tensor.kind() != target_kind {
                tensor.to_kind(target_kind)
            } else {
                tensor
            };

            results.push((name.clone(), tensor));
        }

        Ok(results)
    }
}


pub struct PytorchStreamReader {
    pub path: PathBuf,
}

impl PytorchStreamReader {
    pub fn new(path: &Path) -> Self {
        Self { path: path.to_path_buf() }
    }

    pub fn iter_tensors(&self, dtype_map: &HashMap<String, Kind>) -> Vec<(String, Tensor)> {
        let mut state = tch::nn::VarStore::new(Device::Cpu);
        state.load(&self.path).unwrap();
        state
            .variables()
            .into_iter()
            .map(|(name, tensor)| {
                let target = dtype_map.get(&name).copied().unwrap_or(tensor.kind());
                let tensor = if tensor.kind() != target {
                    tensor.to_kind(target)
                } else {
                    tensor
                };
                (name, tensor)
            })
            .collect()
    }
}

pub enum StreamReader {
    Safetensors(SafetensorsStreamReader),
    Pytorch(PytorchStreamReader),
}

impl StreamReader {
    pub fn from_path(path: &Path) -> std::io::Result<Self> {
        if path.extension().and_then(|e| e.to_str()) == Some("safetensors") {
            Ok(Self::Safetensors(SafetensorsStreamReader::new(path)?))
        } else {
            Ok(Self::Pytorch(PytorchStreamReader::new(path)))
        }
    }

    pub fn iter_tensors(&self, dtype_map: &HashMap<String, Kind>) -> Vec<(String, Tensor)> {
        match self {
            Self::Safetensors(r) => r.iter_tensors(dtype_map).unwrap_or_default(),
            Self::Pytorch(r) => r.iter_tensors(dtype_map),
        }
    }
}


pub struct StreamingModelLoader {
    pub budget: ChunkBudget,
    pub device: Device,
    pub target_kind: Option<Kind>,
    pub loaded_names: HashSet<String>,
}

impl StreamingModelLoader {
    pub fn new(max_chunk_mb: f64, device: Device, target_kind: Option<Kind>) -> Self {
        Self {
            budget: ChunkBudget::new((max_chunk_mb * MB) as usize),
            device,
            target_kind,
            loaded_names: HashSet::new(),
        }
    }

    pub fn group_by_chunk(&self, named_params: &[(String, usize)]) -> Vec<Vec<String>> {
        let mut chunks: Vec<Vec<String>> = Vec::new();
        let mut current: Vec<String> = Vec::new();
        let mut current_bytes: usize = 0;

        for (name, size) in named_params {
            if current_bytes + size > self.budget.max_bytes && !current.is_empty() {
                chunks.push(current.clone());
                current.clear();
                current_bytes = 0;
            }
            current.push(name.clone());
            current_bytes += size;
        }
        if !current.is_empty() {
            chunks.push(current);
        }
        chunks
    }

    pub fn load_from_file(
        &mut self,
        vs: &mut tch::nn::VarStore,
        weight_file: &Path,
        verbose: bool,
    ) {
        let reader = StreamReader::from_path(weight_file).unwrap();
        let variables = vs.variables();
        let param_count = variables.len();

        let dtype_map: HashMap<String, Kind> = if let Some(kind) = self.target_kind {
            variables.keys().map(|n| (n.clone(), kind)).collect()
        } else {
            HashMap::new()
        };

        let named_sizes: Vec<(String, usize)> = variables
            .iter()
            .map(|(n, t)| (n.clone(), tensor_size_bytes(t)))
            .collect();
        let chunks = self.group_by_chunk(&named_sizes);

        let mut name_to_chunk: HashMap<String, usize> = HashMap::new();
        for (chunk_idx, chunk) in chunks.iter().enumerate() {
            for name in chunk {
                name_to_chunk.insert(name.clone(), chunk_idx);
            }
        }

        let mut current_chunk_idx: Option<usize> = None;
        let mut pending: HashMap<String, Tensor> = HashMap::new();

        let flush_chunk = |pending: &mut HashMap<String, Tensor>,
                           vs: &mut tch::nn::VarStore,
                           loaded_names: &mut HashSet<String>,
                           budget: &ChunkBudget,
                           device: Device,
                           verbose: bool,
                           param_count: usize| {
            let vars = vs.variables();
            for (name, tensor) in pending.iter() {
                if let Some(param) = vars.get(name) {
                    tch::no_grad(|| {
                        param.copy_(&tensor.to_device(device));
                    });
                    loaded_names.insert(name.clone());
                }
            }
            if verbose {
                let loaded = loaded_names.len();
                print!(
                    "\r  Loading: {}/{} params ({:.1}%) | chunk budget: {:.0}/{:.0} MB",
                    loaded,
                    param_count,
                    100.0 * loaded as f64 / param_count as f64,
                    budget.used_mb(),
                    budget.max_mb(),
                );
            }
            pending.clear();
            budget.reset();
        };

        for (name, tensor) in reader.iter_tensors(&dtype_map) {
            if !variables.contains_key(&name) {
                continue;
            }
            let chunk_idx = *name_to_chunk.get(&name).unwrap_or(&0);
            if current_chunk_idx.map_or(false, |c| c != chunk_idx) && !pending.is_empty() {
                flush_chunk(
                    &mut pending,
                    vs,
                    &mut self.loaded_names,
                    &self.budget,
                    self.device,
                    verbose,
                    param_count,
                );
            }
            current_chunk_idx = Some(chunk_idx);
            self.budget.consume(tensor_size_bytes(&tensor));
            pending.insert(name, tensor);
        }

        if !pending.is_empty() {
            flush_chunk(
                &mut pending,
                vs,
                &mut self.loaded_names,
                &self.budget,
                self.device,
                verbose,
                param_count,
            );
        }

        if verbose {
            println!(
                "\n  Done. Loaded {}/{} parameters.",
                self.loaded_names.len(),
                param_count,
            );
        }
    }

    pub fn load_from_directory(
        &mut self,
        vs: &mut tch::nn::VarStore,
        directory: &Path,
        pattern: &str,
        verbose: bool,
    ) {
        let mut files: Vec<PathBuf> = glob::glob(
            directory.join(pattern).to_str().unwrap()
        )
        .unwrap()
        .filter_map(|e| e.ok())
        .collect();

        if files.is_empty() {
            files = glob::glob(directory.join("*.bin").to_str().unwrap())
                .unwrap()
                .filter_map(|e| e.ok())
                .collect();
        }

        if files.is_empty() {
            panic!("No weight files found in {}", directory.display());
        }

        files.sort();

        if verbose {
            println!("Found {} weight file(s) in {}", files.len(), directory.display());
        }

        for (i, file) in files.iter().enumerate() {
            if verbose {
                println!("[{}/{}] {}", i + 1, files.len(), file.file_name().unwrap().to_str().unwrap());
            }
            self.load_from_file(vs, file, verbose);
        }
    }
}


pub struct LayerStreamedInference {
    pub config: ModelConfig,
    pub weight_dir: PathBuf,
    pub device: Device,
    pub offload_device: Device,
    pub max_active_layers: usize,
    pub model: CustomLLM,
    pub active_layers: HashMap<usize, ()>,
}

impl LayerStreamedInference {
    pub fn new(
        vs: &tch::nn::Path,
        config: ModelConfig,
        weight_dir: &Path,
        device: Device,
        offload_device: Device,
        max_active_layers: usize,
    ) -> Self {
        let model = CustomLLM::new(vs, config.clone());
        Self {
            config,
            weight_dir: weight_dir.to_path_buf(),
            device,
            offload_device,
            max_active_layers,
            model,
            active_layers: HashMap::new(),
        }
    }

    pub fn ensure_layer_on_device(&mut self, layer_idx: usize) {
        if !self.active_layers.contains_key(&layer_idx) {
            if self.active_layers.len() >= self.max_active_layers {
                if let Some(&evict_idx) = self.active_layers.keys().next() {
                    self.model.layers[evict_idx]
                        .self_attn
                        .rope
                        .inv_freq = self.model.layers[evict_idx]
                        .self_attn
                        .rope
                        .inv_freq
                        .to_device(self.offload_device);
                    self.active_layers.remove(&evict_idx);
                }
            }
            self.active_layers.insert(layer_idx, ());
        }
    }

    pub fn generate_step(
        &mut self,
        input_ids: &Tensor,
        past_key_values: Option<Vec<Option<(Tensor, Tensor)>>>,
    ) -> (Tensor, Vec<Option<(Tensor, Tensor)>>) {
        let shape = input_ids.size();
        let (_bsz, seq_len) = (shape[0], shape[1]);

        let offset = past_key_values
            .as_ref()
            .and_then(|pkvs| pkvs.first())
            .and_then(|pkv| pkv.as_ref())
            .map(|(k, _)| k.size()[2])
            .unwrap_or(0);

        let position_ids = Tensor::arange_start(
            offset,
            offset + seq_len,
            (Kind::Int64, self.device),
        )
        .unsqueeze(0);

        let input_on_device = input_ids.to_device(self.device);
        let mut hidden = self.model.embed_tokens.forward(&input_on_device);

        let num_layers = self.config.num_hidden_layers as usize;
        let mut new_past: Vec<Option<(Tensor, Tensor)>> = Vec::new();

        for i in 0..num_layers {
            self.ensure_layer_on_device(i);
            let past = past_key_values
                .as_ref()
                .and_then(|pkvs| pkvs.get(i))
                .and_then(|pkv| pkv.as_ref())
                .map(|(k, v)| (k.shallow_clone(), v.shallow_clone()));

            let (next_hidden, layer_past, _) = self.model.layers[i].forward(
                &hidden,
                &position_ids,
                None,
                past,
                true,
            );
            hidden = next_hidden;
            new_past.push(layer_past);
        }

        let hidden = self.model.norm.forward(&hidden);
        let logits = match &self.model.lm_head {
            Some(head) => head.forward(&hidden),
            None => hidden.matmul(&self.model.embed_tokens.ws.transpose(0, 1)),
        };

        (logits, new_past)
    }
}

Writing src/streaming.rs


In [ ]:
%%writefile src/data.rs
#![allow(warnings)]

use std::collections::{HashMap, HashSet};
use std::path::PathBuf;
use tiktoken_rs::{get_bpe_from_model, CoreBPE};
use hf_hub::api::sync::Api;
use serde::Deserialize;
use rand::seq::SliceRandom;

pub const MAX_SEQ_LEN: usize = 512;
const MAX_SAMPLES: usize = 50000;

const DATASET_MIX_OASST1: f64 = 0.70;
const DATASET_MIX_EVERYDAY: f64 = 0.20;
const DATASET_MIX_DAILY_DIALOG: f64 = 0.10;


#[derive(Debug, Deserialize)]
pub struct OasstMessage {
    pub message_id: String,
    pub parent_id: Option<String>,
    pub role: String,
    pub text: String,
    pub lang: String,
}

#[derive(Debug, Deserialize)]
pub struct EverydayMessage {
    pub role: String,
    pub content: String,
}

#[derive(Debug, Deserialize)]
pub struct EverydayRow {
    pub messages: Vec<EverydayMessage>,
}

#[derive(Debug, Deserialize)]
pub struct DailyDialogRow {
    pub dialog: Vec<String>,
}


pub fn oasst1_to_threads(messages: Vec<OasstMessage>) -> Vec<String> {
    let mut msg_by_id: HashMap<String, OasstMessage> = HashMap::new();
    let mut children: HashMap<String, Vec<String>> = HashMap::new();

    for msg in messages {
        if let Some(ref parent_id) = msg.parent_id {
            children
                .entry(parent_id.clone())
                .or_default()
                .push(msg.message_id.clone());
        }
        msg_by_id.insert(msg.message_id.clone(), msg);
    }

    let mut threads: Vec<String> = Vec::new();

    fn build_thread(
        msg_id: &str,
        history: Vec<String>,
        msg_by_id: &HashMap<String, OasstMessage>,
        children: &HashMap<String, Vec<String>>,
        threads: &mut Vec<String>,
    ) {
        let msg = match msg_by_id.get(msg_id) {
            Some(m) => m,
            None => return,
        };
        if msg.lang != "en" {
            return;
        }
        let role = if msg.role == "prompter" { "User" } else { "Assistant" };
        let line = format!("{}: {}", role, msg.text.trim());
        let mut new_history = history;
        new_history.push(line);

        let kids = children.get(msg_id);
        match kids {
            None => threads.push(new_history.join("\n")),
            Some(kid_ids) => {
                for kid_id in kid_ids.iter().take(2) {
                    build_thread(kid_id, new_history.clone(), msg_by_id, children, threads);
                }
            }
        }
    }

    let root_ids: Vec<String> = msg_by_id
        .values()
        .filter(|m| m.parent_id.is_none())
        .map(|m| m.message_id.clone())
        .collect();

    for root_id in root_ids {
        build_thread(&root_id, vec![], &msg_by_id, &children, &mut threads);
    }

    threads
}


pub fn everyday_to_threads(rows: Vec<EverydayRow>) -> Vec<String> {
    rows.into_iter()
        .filter_map(|row| {
            let turns: Vec<String> = row
                .messages
                .iter()
                .map(|m| {
                    let role = if m.role == "user" { "User" } else { "Assistant" };
                    format!("{}: {}", role, m.content.trim())
                })
                .collect();
            if turns.is_empty() {
                None
            } else {
                Some(turns.join("\n"))
            }
        })
        .collect()
}


pub fn daily_dialog_to_threads(rows: Vec<DailyDialogRow>) -> Vec<String> {
    let roles = ["User", "Assistant"];
    rows.into_iter()
        .filter_map(|row| {
            let turns: Vec<String> = row
                .dialog
                .iter()
                .enumerate()
                .map(|(i, utt)| format!("{}: {}", roles[i % 2], utt.trim()))
                .collect();
            if turns.is_empty() {
                None
            } else {
                Some(turns.join("\n"))
            }
        })
        .collect()
}


pub fn load_jsonl<T: for<'de> Deserialize<'de>>(path: &PathBuf) -> Vec<T> {
    let content = std::fs::read_to_string(path).unwrap();
    content
        .lines()
        .filter(|l| !l.trim().is_empty())
        .filter_map(|line| serde_json::from_str(line).ok())
        .collect()
}


pub fn download_dataset(hf_token: &str, repo_id: &str, filename: &str) -> PathBuf {
    let api = Api::new().unwrap();
    let repo = api.dataset(repo_id.to_string());
    repo.download(filename).unwrap()
}


pub fn load_all_datasets(hf_token: &str) -> Vec<String> {
    let n_oasst = (MAX_SAMPLES as f64 * DATASET_MIX_OASST1) as usize;
    let n_everyday = (MAX_SAMPLES as f64 * DATASET_MIX_EVERYDAY) as usize;
    let n_daily = (MAX_SAMPLES as f64 * DATASET_MIX_DAILY_DIALOG) as usize;

    let mut all_texts: Vec<String> = Vec::new();

    println!("Loading datasets...");

    println!("  [1/3] OpenAssistant/oasst1 → target {} threads", n_oasst);
    let oasst_path = download_dataset(hf_token, "OpenAssistant/oasst1", "data/train-00000-of-00001.parquet");
    let oasst_messages: Vec<OasstMessage> = load_jsonl(&oasst_path);
    let mut oasst_threads = oasst1_to_threads(oasst_messages);
    oasst_threads.truncate(n_oasst);
    println!("        Got {} English threads", oasst_threads.len());
    all_texts.extend(oasst_threads);

    println!("  [2/3] everyday-conversations → target {} threads", n_everyday);
    let everyday_path = download_dataset(hf_token, "Ichsan2895/everyday-conversations-llama3.1-format", "data/train-00000-of-00001.parquet");
    let everyday_rows: Vec<EverydayRow> = load_jsonl(&everyday_path);
    let mut everyday_threads = everyday_to_threads(everyday_rows);
    everyday_threads.truncate(n_everyday);
    println!("        Got {} threads", everyday_threads.len());
    all_texts.extend(everyday_threads);

    println!("  [3/3] daily_dialog → target {} threads", n_daily);
    let daily_path = download_dataset(hf_token, "daily_dialog", "data/train-00000-of-00001.parquet");
    let daily_rows: Vec<DailyDialogRow> = load_jsonl(&daily_path);
    let mut daily_threads = daily_dialog_to_threads(daily_rows);
    daily_threads.truncate(n_daily);
    println!("        Got {} threads", daily_threads.len());
    all_texts.extend(daily_threads);

    println!("\nSelesai! Total sampel: {}", all_texts.len());
    all_texts
}


pub fn shuffle_texts(texts: &mut Vec<String>) {
    let mut rng = rand::thread_rng();
    texts.shuffle(&mut rng);
}


pub struct Tokenizer {
    pub bpe: CoreBPE,
    pub pad_token_id: u32,
    pub eos_token_id: u32,
    pub vocab_size: usize,
}

impl Tokenizer {
    pub fn new() -> Self {
        println!("Loading tokenizer (GPT-2 BPE)...");
        let bpe = get_bpe_from_model("gpt2").unwrap();
        let eos_token_id = 50256u32;
        println!("Vocab size: 50257");
        Self {
            bpe,
            pad_token_id: eos_token_id,
            eos_token_id,
            vocab_size: 50257,
        }
    }

    pub fn encode(&self, text: &str, max_len: usize) -> Vec<u32> {
        let mut tokens = self.bpe.encode_ordinary(text);
        tokens.push(self.eos_token_id as usize);
        tokens.truncate(max_len);
        tokens.iter().map(|&t| t as u32).collect()
    }

    pub fn encode_padded(&self, text: &str, max_len: usize) -> Vec<u32> {
        let mut tokens = self.encode(text, max_len);
        while tokens.len() < max_len {
            tokens.push(self.pad_token_id);
        }
        tokens
    }
}


pub fn print_gpu_info() {
    if tch::Cuda::is_available() {
        println!("GPU: CUDA available");
        println!("GPU count: {}", tch::Cuda::device_count());
    } else {
        println!("GPU: NOT AVAILABLE");
    }
}

Writing src/data.rs


In [ ]:
%%writefile src/main.rs
#![allow(warnings)]

mod long_rope_position;
mod hybrid_mla;
mod expert;
mod model;
mod util;
mod saver;
mod streaming;
mod data;
mod model_config;

use std::collections::HashMap;
use std::path::{Path, PathBuf};
use std::sync::Arc;
use rand::seq::SliceRandom;
use rand::thread_rng;
use tch::{nn, nn::OptimizerConfig, Device, Kind, Tensor};
use tiktoken_rs::cl100k_base;
use glob::glob;

use model::CustomLLM;
use model_config::ModelConfig;
use util::*;
use saver::*;
use streaming::*;

const BATCH_SIZE: usize = 2;
const GRAD_ACCUM: usize = 8;
const NUM_EPOCHS: usize = 3;
const LEARNING_RATE: f64 = 3e-4;
const SAVE_EVERY: usize = 500;
const AUX_LOSS_COEF: f64 = 0.01;

struct ChatDataset {
    pub encodings: Vec<Tensor>,
}

impl ChatDataset {
    fn new(texts: &[String], tokenizer: &tiktoken_rs::CoreBPE, max_len: usize, pad_token_id: i64) -> Self {
        let encodings = texts.iter().map(|text| {
            let mut ids: Vec<i64> = tokenizer
                .encode_with_special_tokens(text)
                .into_iter()
                .map(|x| x as i64)
                .collect();
            ids.truncate(max_len);
            let pad_len = max_len.saturating_sub(ids.len());
            ids.extend(vec![pad_token_id; pad_len]);
            Tensor::from_slice(&ids)
        }).collect();
        Self { encodings }
    }

    fn len(&self) -> usize {
        self.encodings.len()
    }

    fn get(&self, idx: usize) -> Tensor {
        self.encodings[idx].shallow_clone()
    }
}

pub struct ChatDataloader {
    pub dataset: Arc<ChatDataset>,
    pub batch_size: usize,
    pub shuffle: bool,
    pub drop_last: bool,
    pub indices: Vec<usize>,
    pub pos: usize,
}

impl ChatDataloader {
    fn new(dataset: Arc<ChatDataset>, batch_size: usize, shuffle: bool, drop_last: bool) -> Self {
        let indices: Vec<usize> = (0..dataset.len()).collect();
        Self { dataset, batch_size, shuffle, drop_last, indices, pos: 0 }
    }

    fn reset(&mut self) {
        self.pos = 0;
        if self.shuffle {
            self.indices.shuffle(&mut thread_rng());
        }
    }

    fn len(&self) -> usize {
        if self.drop_last {
            self.dataset.len() / self.batch_size
        } else {
            (self.dataset.len() + self.batch_size - 1) / self.batch_size
        }
    }

    fn next_batch(&mut self) -> Option<Tensor> {
        let remaining = self.indices.len().saturating_sub(self.pos);
        if remaining == 0 || (self.drop_last && remaining < self.batch_size) {
            return None;
        }
        let end = (self.pos + self.batch_size).min(self.indices.len());
        let batch_indices = &self.indices[self.pos..end];
        self.pos = end;
        let tensors: Vec<Tensor> = batch_indices.iter()
            .map(|&i| self.dataset.get(i))
            .collect();
        Some(Tensor::stack(&tensors, 0))
    }
}

fn save_checkpoint(
    step: usize,
    epoch: usize,
    loss: f64,
    vs: &nn::VarStore,
    checkpoint_dir: &str,
) {
    let path = format!("{}/step_{:06}", checkpoint_dir, step);
    std::fs::create_dir_all(&path).unwrap();
    save_model_sharded(vs, Path::new(&path), 500.0, Some(Kind::BFloat16), false).unwrap();
    Tensor::save_multi(
        &[
            ("step", &Tensor::from(step as i64)),
            ("epoch", &Tensor::from(epoch as i64)),
            ("loss", &Tensor::from(loss as f32)),
        ],
        format!("{}/training_state.pt", path),
    ).unwrap();
    println!("  Checkpoint saved -> {}", path);
}

fn load_latest_checkpoint(
    checkpoint_dir: &str,
    vs: &mut nn::VarStore,
    device: Device,
) -> (usize, usize) {
    let pattern = format!("{}/step_*", checkpoint_dir);
    let mut checkpoints: Vec<PathBuf> = glob(&pattern)
        .unwrap()
        .filter_map(|e| e.ok())
        .collect();
    checkpoints.sort();

    let latest = match checkpoints.last() {
        Some(p) => p.clone(),
        None => return (0, 0),
    };

    let state_path = latest.join("training_state.pt");
    if !state_path.exists() {
        return (0, 0);
    }

    let named = Tensor::load_multi(state_path.to_str().unwrap()).unwrap();
    let state_map: HashMap<String, Tensor> = named.into_iter().collect();

    let start_step: usize = state_map["step"].int64_value(&[]) as usize;
    let start_epoch: usize = state_map["epoch"].int64_value(&[]) as usize;
    let loss: f64 = state_map["loss"].double_value(&[]);

    let mut loader = StreamingModelLoader::new(500.0, device, Some(Kind::Float));
    loader.load_from_directory(vs, latest.as_path(), "*.safetensors", false);

    println!("  Resumed from step {}, epoch {}, loss {:.4}", start_step, start_epoch, loss);
    (start_step, start_epoch)
}

fn chat(
    prompt: &str,
    model: &mut CustomLLM,
    tokenizer: &tiktoken_rs::CoreBPE,
    device: Device,
    max_new: usize,
    eos_token_id: i64,
) -> String {
    let text = format!("User: {}\nAssistant:", prompt);
    let ids: Vec<i64> = tokenizer
        .encode_with_special_tokens(&text)
        .into_iter()
        .map(|x| x as i64)
        .collect();
    let input = Tensor::from_slice(&ids).unsqueeze(0).to(device);

    let mut tokens: Vec<i64> = vec![];
    let mut past_key_values: Option<Vec<Option<(Tensor, Tensor)>>> = None;
    let mut cur_input = input.shallow_clone();

    for _ in 0..max_new {
        let (logits, new_past, _) = model.forward(&cur_input, None, None, past_key_values.as_ref(), true, false);
        past_key_values = new_past;
        let next_token = logits.select(1, -1).argmax(-1, false).int64_value(&[0]);
        if next_token == eos_token_id {
            break;
        }
        tokens.push(next_token);
        let decoded_u32: Vec<u32> = tokens.iter().map(|&x| x as u32).collect();
        let decoded = tokenizer.decode(decoded_u32).unwrap();
        if decoded.contains("\nUser:") {
            break;
        }
        cur_input = Tensor::from_slice(&[next_token]).unsqueeze(0).to(device);
    }

    let final_u32: Vec<u32> = tokens.iter().map(|&x| x as u32).collect();
    tokenizer.decode(final_u32).unwrap_or_default()
}

fn medium_config() -> ModelConfig {
    model_config::preset_4b()
}

fn small_config() -> ModelConfig {
    ModelConfig {
        hidden_size: 1024,
        intermediate_size: 4096,
        num_hidden_layers: 16,
        num_attention_heads: 8,
        num_key_value_heads: 4,
        head_dim: 128,
        kv_lora_rank: 256,
        q_lora_rank: 384,
        num_experts: 32,
        num_shared_experts: 2,
        num_experts_per_tok: 4,
        moe_intermediate_size: 512,
        ..ModelConfig::default()
    }
}

fn load_texts() -> Vec<String> {
    let hf_token = std::env::var("HF_TOKEN").unwrap_or_default();
    data::load_all_datasets(&hf_token)
}

fn main() {
    let device = if tch::Cuda::is_available() { Device::Cuda(0) } else { Device::Cpu };

    let vram_gb = if tch::Cuda::is_available() {
        tch::Cuda::device_count();
        if let Ok(props) = std::panic::catch_unwind(|| tch::Cuda::device_count()) {
            8.0_f64
        } else {
            0.0
        }
    } else {
        0.0
    };

    let config = if vram_gb >= 30.0 { medium_config() } else { small_config() };
    println!("VRAM={:.1}GB -> using {} config", vram_gb, if vram_gb >= 30.0 { "MEDIUM" } else { "SMALL" });

    let mut vs = nn::VarStore::new(device);
    let mut model = CustomLLM::new(&vs.root(), config.clone());

    let stats = count_parameters(&vs);
    println!("Model: {:.1}M total params", stats["total"] as f64 / 1e6);
    println!("Active/tok: {:.1}M", estimate_active_params(&config)["active_params"] as f64 / 1e6);

    let tokenizer = cl100k_base().unwrap();
    let pad_token_id: i64 = 0;
    let eos_token_id: i64 = tokenizer.encode_with_special_tokens("<|endoftext|>")[0] as i64;

    let all_texts: Vec<String> = load_texts();
    println!("Tokenizing dataset...");
    let dataset = Arc::new(ChatDataset::new(&all_texts, &tokenizer, data::MAX_SEQ_LEN, pad_token_id));

    let n_train = (dataset.len() as f64 * 0.95) as usize;
    let n_val = dataset.len() - n_train;

    let all_indices: Vec<usize> = {
        let mut v: Vec<usize> = (0..dataset.len()).collect();
        v.shuffle(&mut thread_rng());
        v
    };
    let train_indices = all_indices[..n_train].to_vec();
    let val_indices = all_indices[n_train..].to_vec();

    let train_ds = Arc::new(ChatDataset {
        encodings: train_indices.iter().map(|&i| dataset.get(i)).collect(),
    });
    let val_ds = Arc::new(ChatDataset {
        encodings: val_indices.iter().map(|&i| dataset.get(i)).collect(),
    });

    let mut train_loader = ChatDataloader::new(train_ds, BATCH_SIZE, true, true);
    let mut val_loader = ChatDataloader::new(val_ds, BATCH_SIZE, false, false);

    let steps_per_epoch = train_loader.len() / GRAD_ACCUM;
    let max_steps = steps_per_epoch * NUM_EPOCHS;
    println!("Train: {} | Val: {}", n_train, n_val);
    println!("Steps/epoch: {} | Total steps: {}", steps_per_epoch, max_steps);

    let mut optimizer = build_optimizer(&vs, LEARNING_RATE, 0.1, 0.9, 0.95);
    let scheduler = LRScheduler::new(LEARNING_RATE, max_steps, max_steps / 20, 0.1);

    let checkpoint_dir = "checkpoints";
    println!("Checking for existing checkpoint...");
    let (mut global_step, start_epoch) = load_latest_checkpoint(checkpoint_dir, &mut vs, device);
    if global_step == 0 {
        println!("  No checkpoint found, training from scratch.");
    }

    for epoch in start_epoch..NUM_EPOCHS {
        println!("\n=== Epoch {}/{} ===", epoch + 1, NUM_EPOCHS);

        vs.set_kind(Kind::Float);
        train_loader.reset();
        optimizer.zero_grad();

        let mut total_lm = 0.0_f64;
        let mut total_aux = 0.0_f64;
        let mut batch_count = 0_usize;
        let mut step = 0_usize;

        while let Some(batch) = train_loader.next_batch() {
            let batch = batch.to(device);
            let mut labels = batch.shallow_clone();
            let _ = labels.where_self(&labels.ne(pad_token_id), &Tensor::from(-100_i64));

            let (loss, metrics) = tch::autocast(device == Device::Cuda(0), || {
                compute_loss(&mut model, &batch, &labels, AUX_LOSS_COEF)
            });
            let scaled_loss = &loss / GRAD_ACCUM as f64;
            scaled_loss.backward();

            total_lm += metrics["lm_loss"];
            total_aux += metrics["aux_loss"];
            batch_count += 1;

            if (step + 1) % GRAD_ACCUM == 0 {
                let _ = nn::clip_grad_norm(&vs.trainable_variables(), 1.0);
                optimizer.step();
                scheduler.step(&mut optimizer, global_step);
                optimizer.zero_grad();
                global_step += 1;

                let avg_lm = total_lm / batch_count as f64;
                let avg_aux = total_aux / batch_count as f64;
                let lr_now = scheduler.get_lr(global_step);

                println!(
                    "step={} lm={:.3} aux={:.3} lr={:.1e}",
                    global_step, avg_lm, avg_aux, lr_now
                );

                if global_step % SAVE_EVERY == 0 {
                    save_checkpoint(global_step, epoch, avg_lm, &vs, checkpoint_dir);
                }

                total_lm = 0.0;
                total_aux = 0.0;
                batch_count = 0;
            }

            step += 1;
        }

        let mut val_loss = 0.0_f64;
        let mut val_steps = 0_usize;

        tch::no_grad(|| {
            while let Some(vbatch) = val_loader.next_batch() {
                let vbatch = vbatch.to(device);
                let mut vlabels = vbatch.shallow_clone();
                let _ = vlabels.where_self(&vlabels.ne(pad_token_id), &Tensor::from(-100_i64));
                let (_, vmetrics) = tch::autocast(device == Device::Cuda(0), || {
                    compute_loss(&mut model, &vbatch, &vlabels, AUX_LOSS_COEF)
                });
                val_loss += vmetrics["lm_loss"];
                val_steps += 1;
            }
        });

        let avg_val = val_loss / val_steps.max(1) as f64;
        let perplexity = avg_val.exp();
        println!("  Val LM loss: {:.4} | perplexity: {:.2}", avg_val, perplexity);

        save_checkpoint(global_step, epoch + 1, avg_val, &vs, checkpoint_dir);
    }

    println!("\nTraining complete!");

    println!("{}", chat("Hi, how are you?", &mut model, &tokenizer, device, 80, eos_token_id));
    println!("{}", chat("What did you do today?", &mut model, &tokenizer, device, 80, eos_token_id));
    println!("{}", chat("Tell me something interesting.", &mut model, &tokenizer, device, 80, eos_token_id));

    let final_dir = "output/final_model";
    save_model_sharded(&vs, Path::new(final_dir), 500.0, Some(Kind::BFloat16), true).unwrap();
    println!("\nFinal model saved to {}", final_dir);
}

Overwriting src/main.rs


In [ ]:

!rm -rf Cargo.lock
!cargo clean

     Removed 0 files


In [ ]:
!cargo run --release

    Updating crates.io index
     Locking 268 packages to latest Rust 1.96.0 compatible versions
      Adding generic-array v0.14.7 (available: v0.14.9)
  Downloaded chacha20 v0.10.1
  Downloaded cpufeatures v0.3.0
  Downloaded rand_core v0.10.1
  Downloaded getrandom v0.4.3
  Downloaded rand v0.10.1
   Compiling libc v0.2.186
   Compiling proc-macro2 v1.0.106
   Compiling quote v1.0.46
   Compiling unicode-ident v1.0.24
   Compiling syn v2.0.118
   Compiling jobserver v0.1.34
   Compiling shlex v2.0.1
   Compiling find-msvc-tools v0.1.9
   Compiling cc v1.2.65
   Compiling pkg-config v0.3.33
   Compiling version_check v0.9.5
   Compiling cfg-if v1.0.4
   Compiling generic-array v0.14.7
   Compiling synstructure v0.13.2
   Compiling memchr v2.8.2
   Compiling zerofrom-derive v0.1.7
   Compiling zstd-sys v2.0.16+zstd.1.5.7
   Compiling typenum v1.20.1
   Compiling zerofrom v0.1.8
   Compiling yoke-derive v0.8.2
   Compiling itoa v1.0.18
   Compiling stable_deref_trait v1.2.1
   Compilin

In [ ]:
!(tail -n +1 Cargo.toml; tail -n +1 $(find src -type f)) > code.txt

tail: cannot open 'Cargo.toml' for reading: No such file or directory
find: ‘src’: No such file or directory
^C


In [ ]:
!ls

code.txt  sample_data


In [ ]:
from google.colab import files
files.download('code.txt')